
# XQuality — Incremental layer contribution analysis

This notebook does **not** try to prove that every layer monotonically improves final F1.

Instead, it asks a cleaner ablation question:

> What does each layer add compared with the previous cumulative evidence pool?

It measures:

1. **New information added per layer**
2. **Reinforced information already seen in earlier layers**
3. **New information that aligns with gold triples**
4. **New information that aligns with the native NeoOLAF final export**
5. **New uncertain / reference-unmatched information**
6. **New likely noise**: reference-unmatched triples that appear only once and are not reused by later layers
7. **Cumulative coverage of the native final export**

This is designed to answer:

> Does the cumulative layer evidence increasingly contain the material needed to produce the final NeoOLAF KG?


In [ ]:

from pathlib import Path
import sys
import json
import re
import math
import unicodedata
from difflib import SequenceMatcher
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start=None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "neoolaf").exists():
            return p
    for p in candidates:
        if (p / "pyproject.toml").exists() and (p / "examples" / "XQualityMachine32").exists():
            return p
    raise RuntimeError("Could not find NeoOLAF project root. Run this notebook from inside the NeoOLAF repository.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

EXAMPLE_ROOT = PROJECT_ROOT / "examples" / "XQualityMachine32"
RUNS_ROOT = EXAMPLE_ROOT / "runs"

PREFIX_RUNS_DIR = RUNS_ROOT / "prefix_stop_after_layer_llm_finalization_exact_eval"
GOLD_PATH = PROJECT_ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"

# Leave as None to auto-detect. If auto-detection chooses the wrong folder, set manually.
FINAL_EXPORT_DIR = None

OUTPUT_DIR = RUNS_ROOT / "incremental_layer_contribution_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREFIX_RUNS_DIR:", PREFIX_RUNS_DIR)
print("GOLD_PATH:", GOLD_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

assert PREFIX_RUNS_DIR.exists(), f"Missing prefix run directory: {PREFIX_RUNS_DIR}"
assert GOLD_PATH.exists(), f"Missing gold file: {GOLD_PATH}"



## Configuration

The matching below is **not the official NeoOLAF evaluator**. It is an **incremental alignment analysis**.

It uses predicate-aware approximate matching to decide whether two triples are close enough to represent the same information.

You can tune the thresholds below. If matching is too strict, cumulative final-export coverage will be underestimated. If it is too loose, uncertain/noise will be underestimated.


In [ ]:

# Matching thresholds.
SUBJECT_SIM_THRESHOLD = 0.72
OBJECT_SIM_THRESHOLD = 0.68

# Approximate clustering settings.
ENABLE_APPROX_CLUSTERING = True
CLUSTER_SUBJECT_SIM_THRESHOLD = 0.82
CLUSTER_OBJECT_SIM_THRESHOLD = 0.78

# If True, same-predicate matching only. Recommended.
REQUIRE_SAME_PREDICATE = True

# Minimum triples in a prefix folder.
MIN_TRIPLES_PER_PREFIX = 1

# Whether to require COMPLETED.json marker for prefix folders.
# Set False if older generated folders do not have marker but have triples.json.
STRICT_COMPLETED_MARKER = False

# If True, unmatched clusters that occur in only one layer are counted as likely noise.
COUNT_SINGLE_LAYER_UNMATCHED_AS_LIKELY_NOISE = True

print("Config loaded.")


## Triple loading utilities

In [ ]:

def load_json_any(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def unwrap_triple_container(obj):
    """Return a list-like collection of triples from common NeoOLAF/prefix JSON shapes."""
    if obj is None:
        return []
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in [
            "triples",
            "relations",
            "candidate_triples",
            "candidate_relation_assertions",
            "kg",
            "data",
            "items",
        ]:
            value = obj.get(key)
            if isinstance(value, list):
                return value
        # Sometimes dictionary values themselves are triples.
        if all(isinstance(v, dict) for v in obj.values()):
            return list(obj.values())
    return []


def first_nonempty(*values):
    for v in values:
        if v is not None:
            s = str(v).strip()
            if s and s.lower() not in {"none", "null", "nan", "?"}:
                return s
    return ""


def normalize_predicate_label(x):
    s = first_nonempty(x)
    if not s:
        return ""
    # If relation has "Pxxx : label", keep the label part when useful.
    if ":" in s:
        left, right = s.split(":", 1)
        if right.strip():
            s = right.strip()
    s = s.upper().strip()
    s = re.sub(r"[^A-Z0-9]+", "_", s).strip("_")

    aliases = {
        "CAUSE": "CAUSES",
        "CAUSES": "CAUSES",
        "CAUSED_BY": "CAUSES",
        "TRIGGER": "TRIGGERS",
        "TRIGGERS": "TRIGGERS",
        "TRIGGERED_BY": "TRIGGERS",
        "REQUIRE": "REQUIRES",
        "REQUIRES": "REQUIRES",
        "REQUIRED_BY": "REQUIRES",
        "REFERENCE": "REFERENCES",
        "REFERENCES": "REFERENCES",
        "REFERS_TO": "REFERENCES",
        "HANDLED_BY": "HANDLED_BY",
        "HANDLEDBY": "HANDLED_BY",
        "HANDLER": "HANDLED_BY",
    }
    return aliases.get(s, s)


def triple_from_record(rec, default_source=""):
    """Normalize many possible triple schemas into a common dict."""
    if not isinstance(rec, dict):
        return None

    subject = first_nonempty(
        rec.get("subject"),
        rec.get("head"),
        rec.get("source"),
        rec.get("subject_label"),
        rec.get("head_label"),
        rec.get("source_label"),
        rec.get("s"),
    )
    predicate = first_nonempty(
        rec.get("predicate"),
        rec.get("relation"),
        rec.get("relation_label"),
        rec.get("predicate_label"),
        rec.get("p"),
    )
    obj = first_nonempty(
        rec.get("object"),
        rec.get("tail"),
        rec.get("target"),
        rec.get("object_label"),
        rec.get("tail_label"),
        rec.get("target_label"),
        rec.get("o"),
    )

    if not subject or not predicate or not obj:
        return None

    evidence = first_nonempty(
        rec.get("evidence"),
        rec.get("support"),
        rec.get("source_text"),
        rec.get("provenance"),
        rec.get("context"),
        rec.get("sentence"),
        rec.get("chunk_text"),
    )

    confidence = rec.get("confidence", rec.get("score", rec.get("support_score", None)))
    try:
        confidence = float(confidence) if confidence is not None else None
    except Exception:
        confidence = None

    return {
        "subject": subject,
        "predicate": normalize_predicate_label(predicate),
        "object": obj,
        "evidence": evidence,
        "confidence": confidence,
        "raw": rec,
        "source_file": default_source,
    }


def load_triples_from_file(path: Path):
    try:
        obj = load_json_any(path)
    except Exception as e:
        print("Could not read", path, "->", repr(e))
        return []
    rows = []
    for rec in unwrap_triple_container(obj):
        t = triple_from_record(rec, default_source=str(path))
        if t is not None:
            rows.append(t)
    return rows


def load_triples_from_folder(folder: Path):
    candidates = [
        "triples.json",
        "kg_inferred.json",
        "kg.json",
        "kg_local.json",
        "relations.json",
        "candidate_triples.json",
    ]
    for name in candidates:
        path = folder / name
        if path.exists():
            triples = load_triples_from_file(path)
            if triples:
                return triples, path
    # fallback: try all json files in folder
    for path in sorted(folder.glob("*.json")):
        triples = load_triples_from_file(path)
        if triples:
            return triples, path
    return [], None


def count_triples_in_folder(folder: Path):
    triples, _ = load_triples_from_folder(Path(folder))
    return len(triples)


## Text normalization and approximate matching

In [ ]:

STOPWORDS = {
    "the", "a", "an", "of", "to", "and", "or", "for", "in", "on", "with", "by", "is", "are", "be", "as",
    "le", "la", "les", "de", "des", "du", "un", "une", "et", "ou", "pour", "dans", "sur", "avec", "par",
    "message", "alarm", "alarme", "error", "fault", "warning", "code",
}


def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(s))
        if not unicodedata.combining(c)
    )


def norm_text(s):
    s = strip_accents(str(s).lower())
    s = re.sub(r"[_/\\\-]+", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def tokens(s):
    return [t for t in norm_text(s).split() if t and t not in STOPWORDS]


def token_jaccard(a, b):
    ta, tb = set(tokens(a)), set(tokens(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def token_containment(a, b):
    ta, tb = set(tokens(a)), set(tokens(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / min(len(ta), len(tb))


def char_ngram_jaccard(a, b, n=3):
    a = norm_text(a)
    b = norm_text(b)
    if len(a) < n or len(b) < n:
        return 0.0
    ga = {a[i:i+n] for i in range(len(a)-n+1)}
    gb = {b[i:i+n] for i in range(len(b)-n+1)}
    if not ga or not gb:
        return 0.0
    return len(ga & gb) / len(ga | gb)


def text_similarity(a, b):
    a0, b0 = norm_text(a), norm_text(b)
    if not a0 or not b0:
        return 0.0
    if a0 == b0:
        return 1.0

    seq = SequenceMatcher(None, a0, b0).ratio()
    jac = token_jaccard(a0, b0)
    cont = token_containment(a0, b0)
    ch3 = char_ngram_jaccard(a0, b0, n=3)

    # For short labels, containment can be too optimistic, so cap it a bit unless sequence also agrees.
    if min(len(tokens(a0)), len(tokens(b0))) <= 2 and seq < 0.60:
        cont = min(cont, 0.65)

    return max(seq, jac, cont, ch3)


def exact_triple_key(t):
    return (
        normalize_predicate_label(t.get("predicate", "")),
        norm_text(t.get("subject", "")),
        norm_text(t.get("object", "")),
    )


def triple_text(t):
    return f"{t.get('subject','')} -- {t.get('predicate','')} --> {t.get('object','')}"


def triple_match_score(a, b):
    pa = normalize_predicate_label(a.get("predicate", ""))
    pb = normalize_predicate_label(b.get("predicate", ""))
    if REQUIRE_SAME_PREDICATE and pa != pb:
        return {
            "match": False,
            "predicate_equal": False,
            "subject_sim": 0.0,
            "object_sim": 0.0,
            "score": 0.0,
        }

    ss = text_similarity(a.get("subject", ""), b.get("subject", ""))
    os = text_similarity(a.get("object", ""), b.get("object", ""))
    score = min(ss, os)
    return {
        "match": (pa == pb or not REQUIRE_SAME_PREDICATE) and ss >= SUBJECT_SIM_THRESHOLD and os >= OBJECT_SIM_THRESHOLD,
        "predicate_equal": pa == pb,
        "subject_sim": ss,
        "object_sim": os,
        "score": score,
    }


## Discover prefix folders and native final export

In [ ]:

def parse_stop_index(folder_name):
    m = re.search(r"prefix_stop_after_(\d+)_", folder_name)
    return int(m.group(1)) if m else None


def discover_prefix_folders(root: Path):
    rows = []
    for folder in sorted(x for x in Path(root).iterdir() if x.is_dir()):
        stop_index = parse_stop_index(folder.name)
        if stop_index is None:
            continue
        if STRICT_COMPLETED_MARKER and not (folder / "COMPLETED.json").exists():
            continue
        triples, triple_file = load_triples_from_folder(folder)
        if len(triples) < MIN_TRIPLES_PER_PREFIX:
            continue
        rows.append({
            "stop_index": stop_index,
            "layer_name": folder.name,
            "folder": folder,
            "triple_file": triple_file,
            "triple_count": len(triples),
        })
    rows = sorted(rows, key=lambda r: r["stop_index"])
    return rows


def discover_final_export_dir():
    if FINAL_EXPORT_DIR is not None:
        p = Path(FINAL_EXPORT_DIR)
        assert p.exists(), f"Manual FINAL_EXPORT_DIR does not exist: {p}"
        return p

    candidates = []
    for path in RUNS_ROOT.rglob("*"):
        if not path.is_file():
            continue
        if path.name not in {"kg_inferred.json", "kg.json", "kg_local.json", "triples.json"}:
            continue
        s = str(path).lower()
        if "prefix_stop_after" in s:
            continue
        if "robust_eval" in s or "cumulative" in s or "olaf" in s:
            continue
        folder = path.parent
        triples = load_triples_from_file(path)
        n = len(triples)
        if n == 0:
            continue
        # prefer something near the known final export size 226
        score = abs(n - 226)
        candidates.append((score, -path.stat().st_mtime, n, folder, path))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: (x[0], x[1]))
    print("Top final export candidates:")
    for score, neg_mtime, n, folder, path in candidates[:10]:
        print(f"  n={n:4d} score={score:4d} file={path}")

    return candidates[0][3]


prefix_folders = discover_prefix_folders(PREFIX_RUNS_DIR)
print(f"Found {len(prefix_folders)} prefix folders")
display(pd.DataFrame([
    {
        "stop_index": r["stop_index"],
        "layer_name": r["layer_name"],
        "triple_count": r["triple_count"],
        "folder": str(r["folder"]),
        "triple_file": str(r["triple_file"]),
    }
    for r in prefix_folders
]))

FINAL_EXPORT_DIR_RESOLVED = discover_final_export_dir()
print("Selected FINAL_EXPORT_DIR:", FINAL_EXPORT_DIR_RESOLVED)

if FINAL_EXPORT_DIR_RESOLVED is None:
    print("WARNING: no native final export detected. Native-final alignment metrics will be skipped.")
else:
    final_triples, final_file = load_triples_from_folder(FINAL_EXPORT_DIR_RESOLVED)
    print("Native final export triple count:", len(final_triples))
    print("Native final export file:", final_file)
    if len(final_triples) != 226:
        print("WARNING: selected final export is not 226 triples. Manually set FINAL_EXPORT_DIR if this is wrong.")


## Load gold and final references

In [ ]:

def load_gold_triples(gold_path: Path):
    obj = load_json_any(gold_path)
    triples = []

    # Try direct container first.
    for rec in unwrap_triple_container(obj):
        t = triple_from_record(rec, default_source=str(gold_path))
        if t is not None:
            triples.append(t)

    # Try nested JSON structures.
    if not triples and isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, list):
                for rec in value:
                    t = triple_from_record(rec, default_source=str(gold_path))
                    if t is not None:
                        triples.append(t)

    # XQuality flat files sometimes use subject/predicate/object in nested triplet fields.
    if not triples and isinstance(obj, list):
        for rec in obj:
            if not isinstance(rec, dict):
                continue
            t = triple_from_record({
                "subject": rec.get("source") or rec.get("subject") or rec.get("head") or rec.get("source_label"),
                "predicate": rec.get("relation") or rec.get("predicate") or rec.get("relation_label"),
                "object": rec.get("target") or rec.get("object") or rec.get("tail") or rec.get("target_label"),
            }, default_source=str(gold_path))
            if t is not None:
                triples.append(t)

    # Deduplicate exact gold triples
    seen = set()
    out = []
    for t in triples:
        k = exact_triple_key(t)
        if k not in seen:
            seen.add(k)
            out.append(t)

    return out


gold_triples = load_gold_triples(GOLD_PATH)
print("Gold triples loaded:", len(gold_triples))
if len(gold_triples) == 0:
    raise RuntimeError("Gold triples are zero. The gold loader did not understand the file. Check GOLD_PATH or adapt load_gold_triples().")

native_final_triples = []
if FINAL_EXPORT_DIR_RESOLVED is not None:
    native_final_triples, native_final_file = load_triples_from_folder(FINAL_EXPORT_DIR_RESOLVED)
    print("Native final triples loaded:", len(native_final_triples))

print("Gold predicate counts:", Counter(t["predicate"] for t in gold_triples))
print("Final predicate counts:", Counter(t["predicate"] for t in native_final_triples))


## Load all layer triples

In [ ]:

layer_triples = {}
all_candidate_triples = []

for rec in tqdm(prefix_folders, desc="Loading layer triples"):
    triples, triple_file = load_triples_from_folder(rec["folder"])
    enriched = []
    for i, t in enumerate(triples):
        tt = dict(t)
        tt["stop_index"] = rec["stop_index"]
        tt["layer_name"] = rec["layer_name"]
        tt["local_index"] = i
        tt["triple_uid"] = f"L{rec['stop_index']:02d}_{i:06d}"
        enriched.append(tt)
    layer_triples[rec["stop_index"]] = enriched
    all_candidate_triples.extend(enriched)

print("Total candidate triples across layers:", len(all_candidate_triples))
print("Layer counts:", {k: len(v) for k, v in layer_triples.items()})


## Cluster equivalent / near-equivalent candidate triples

In [ ]:

def should_cluster(a, b):
    if normalize_predicate_label(a.get("predicate", "")) != normalize_predicate_label(b.get("predicate", "")):
        return False
    ss = text_similarity(a.get("subject", ""), b.get("subject", ""))
    os = text_similarity(a.get("object", ""), b.get("object", ""))
    return ss >= CLUSTER_SUBJECT_SIM_THRESHOLD and os >= CLUSTER_OBJECT_SIM_THRESHOLD


def build_candidate_clusters(triples):
    # First exact dedup groups.
    exact_groups = defaultdict(list)
    for t in triples:
        exact_groups[exact_triple_key(t)].append(t)

    unique_reps = []
    for key, members in exact_groups.items():
        # choose first as representative; keep all members.
        rep = dict(members[0])
        rep["_exact_members"] = members
        unique_reps.append(rep)

    print("Unique exact triples before approximate clustering:", len(unique_reps))

    if not ENABLE_APPROX_CLUSTERING:
        clusters = []
        assignment = {}
        for cid, rep in enumerate(unique_reps):
            members = rep.pop("_exact_members", [rep])
            clusters.append({
                "cluster_id": cid,
                "representative": rep,
                "members": members,
            })
            for m in members:
                assignment[m["triple_uid"]] = cid
        return clusters, assignment

    clusters_by_pred = defaultdict(list)
    clusters = []
    assignment = {}

    for rep in tqdm(unique_reps, desc="Approximate clustering unique triples"):
        pred = normalize_predicate_label(rep.get("predicate", ""))
        members = rep.pop("_exact_members", [rep])

        chosen_cluster = None
        for cid in clusters_by_pred[pred]:
            if should_cluster(rep, clusters[cid]["representative"]):
                chosen_cluster = cid
                break

        if chosen_cluster is None:
            chosen_cluster = len(clusters)
            clusters.append({
                "cluster_id": chosen_cluster,
                "representative": rep,
                "members": [],
            })
            clusters_by_pred[pred].append(chosen_cluster)

        clusters[chosen_cluster]["members"].extend(members)
        for m in members:
            assignment[m["triple_uid"]] = chosen_cluster

    return clusters, assignment


clusters, triple_to_cluster = build_candidate_clusters(all_candidate_triples)
print("Candidate clusters:", len(clusters))

cluster_rows = []
for c in clusters:
    layers = sorted({m["stop_index"] for m in c["members"]})
    rep = c["representative"]
    cluster_rows.append({
        "cluster_id": c["cluster_id"],
        "predicate": rep["predicate"],
        "subject": rep["subject"],
        "object": rep["object"],
        "member_count": len(c["members"]),
        "first_layer": min(layers),
        "last_layer": max(layers),
        "layer_span": max(layers) - min(layers),
        "num_layers_present": len(layers),
        "layers_present": ",".join(map(str, layers)),
        "triple_text": triple_text(rep),
    })

cluster_df = pd.DataFrame(cluster_rows).sort_values(["first_layer", "predicate", "cluster_id"])
display(cluster_df.head(20))
cluster_df.to_csv(OUTPUT_DIR / "candidate_cluster_inventory.csv", index=False)
print("Saved:", OUTPUT_DIR / "candidate_cluster_inventory.csv")


## Match candidate clusters to gold and native final export

In [ ]:

def best_reference_match(query_triple, reference_triples):
    best = None
    best_score = -1.0
    qpred = normalize_predicate_label(query_triple.get("predicate", ""))
    for i, ref in enumerate(reference_triples):
        if REQUIRE_SAME_PREDICATE and normalize_predicate_label(ref.get("predicate", "")) != qpred:
            continue
        score_info = triple_match_score(query_triple, ref)
        if score_info["score"] > best_score:
            best_score = score_info["score"]
            best = (i, ref, score_info)
    if best is None:
        return {
            "matched": False,
            "ref_index": None,
            "score": 0.0,
            "subject_sim": 0.0,
            "object_sim": 0.0,
        }
    i, ref, info = best
    return {
        "matched": bool(info["match"]),
        "ref_index": i if info["match"] else None,
        "best_ref_index": i,
        "score": info["score"],
        "subject_sim": info["subject_sim"],
        "object_sim": info["object_sim"],
        "best_ref_text": triple_text(ref),
    }


cluster_alignment = []

for c in tqdm(clusters, desc="Matching clusters to references"):
    rep = c["representative"]
    gold_m = best_reference_match(rep, gold_triples)
    final_m = best_reference_match(rep, native_final_triples) if native_final_triples else {
        "matched": False,
        "ref_index": None,
        "score": 0.0,
        "subject_sim": 0.0,
        "object_sim": 0.0,
    }

    layers = sorted({m["stop_index"] for m in c["members"]})
    cluster_alignment.append({
        "cluster_id": c["cluster_id"],
        "predicate": rep["predicate"],
        "subject": rep["subject"],
        "object": rep["object"],
        "triple_text": triple_text(rep),
        "first_layer": min(layers),
        "last_layer": max(layers),
        "num_layers_present": len(layers),
        "layers_present": ",".join(map(str, layers)),
        "member_count": len(c["members"]),
        "matches_gold": gold_m["matched"],
        "gold_ref_index": gold_m.get("ref_index"),
        "gold_best_score": gold_m.get("score", 0.0),
        "gold_subject_sim": gold_m.get("subject_sim", 0.0),
        "gold_object_sim": gold_m.get("object_sim", 0.0),
        "gold_best_ref_text": gold_m.get("best_ref_text", ""),
        "matches_native_final": final_m["matched"],
        "native_final_ref_index": final_m.get("ref_index"),
        "native_final_best_score": final_m.get("score", 0.0),
        "native_final_subject_sim": final_m.get("subject_sim", 0.0),
        "native_final_object_sim": final_m.get("object_sim", 0.0),
        "native_final_best_ref_text": final_m.get("best_ref_text", ""),
    })

cluster_alignment_df = pd.DataFrame(cluster_alignment)
cluster_alignment_df["matches_any_reference"] = (
    cluster_alignment_df["matches_gold"] | cluster_alignment_df["matches_native_final"]
)

# A reference-unmatched cluster that appears in only one layer is a plausible noise candidate.
cluster_alignment_df["likely_noise_single_layer"] = (
    (~cluster_alignment_df["matches_any_reference"]) &
    (cluster_alignment_df["num_layers_present"] == 1)
)

# Persistent but not reference-matched means uncertain, not necessarily wrong.
cluster_alignment_df["persistent_unmatched_uncertain"] = (
    (~cluster_alignment_df["matches_any_reference"]) &
    (cluster_alignment_df["num_layers_present"] > 1)
)

display(cluster_alignment_df.head(20))
cluster_alignment_df.to_csv(OUTPUT_DIR / "candidate_cluster_reference_alignment.csv", index=False)
print("Saved:", OUTPUT_DIR / "candidate_cluster_reference_alignment.csv")

print(cluster_alignment_df[[
    "matches_gold",
    "matches_native_final",
    "matches_any_reference",
    "likely_noise_single_layer",
    "persistent_unmatched_uncertain",
]].mean(numeric_only=True))


## Compute cumulative reference coverage and marginal additions per layer

In [ ]:

# Map layer -> set of cluster_ids present in that layer.
layer_cluster_ids = {}
for stop_index, triples in layer_triples.items():
    ids = set()
    for t in triples:
        cid = triple_to_cluster.get(t["triple_uid"])
        if cid is not None:
            ids.add(cid)
    layer_cluster_ids[stop_index] = ids


# Reference index -> clusters that can match it.
gold_ref_to_clusters = defaultdict(set)
final_ref_to_clusters = defaultdict(set)

for _, row in cluster_alignment_df.iterrows():
    cid = int(row["cluster_id"])
    if bool(row["matches_gold"]) and pd.notna(row["gold_ref_index"]):
        gold_ref_to_clusters[int(row["gold_ref_index"])].add(cid)
    if bool(row["matches_native_final"]) and pd.notna(row["native_final_ref_index"]):
        final_ref_to_clusters[int(row["native_final_ref_index"])].add(cid)


seen_clusters = set()
covered_gold_refs = set()
covered_final_refs = set()

summary_rows = []
new_cluster_detail_rows = []

for stop_index in sorted(layer_cluster_ids):
    current = set(layer_cluster_ids[stop_index])
    cumulative_before = set(seen_clusters)
    new_clusters = current - cumulative_before
    reinforced_clusters = current & cumulative_before

    # update cumulative
    seen_clusters |= current

    # covered references after this layer
    for ref_i, cids in gold_ref_to_clusters.items():
        if seen_clusters & cids:
            covered_gold_refs.add(ref_i)
    for ref_i, cids in final_ref_to_clusters.items():
        if seen_clusters & cids:
            covered_final_refs.add(ref_i)

    # first-introduced reference coverage by this layer
    newly_covered_gold = set()
    newly_covered_final = set()

    before_gold = set()
    before_final = set()
    for ref_i, cids in gold_ref_to_clusters.items():
        if cumulative_before & cids:
            before_gold.add(ref_i)
    for ref_i, cids in final_ref_to_clusters.items():
        if cumulative_before & cids:
            before_final.add(ref_i)

    for ref_i, cids in gold_ref_to_clusters.items():
        if (new_clusters & cids) and ref_i not in before_gold:
            newly_covered_gold.add(ref_i)
    for ref_i, cids in final_ref_to_clusters.items():
        if (new_clusters & cids) and ref_i not in before_final:
            newly_covered_final.add(ref_i)

    new_df = cluster_alignment_df[cluster_alignment_df["cluster_id"].isin(new_clusters)].copy()

    n_new = len(new_clusters)
    n_current = len(current)

    def rate(count, denom):
        return float(count) / float(denom) if denom else 0.0

    new_matches_gold = int(new_df["matches_gold"].sum()) if n_new else 0
    new_matches_final = int(new_df["matches_native_final"].sum()) if n_new else 0
    new_matches_any = int(new_df["matches_any_reference"].sum()) if n_new else 0
    new_likely_noise = int(new_df["likely_noise_single_layer"].sum()) if n_new else 0
    new_persistent_unmatched = int(new_df["persistent_unmatched_uncertain"].sum()) if n_new else 0

    layer_name = [r["layer_name"] for r in prefix_folders if r["stop_index"] == stop_index]
    layer_name = layer_name[0] if layer_name else f"layer_{stop_index}"

    summary_rows.append({
        "stop_index": stop_index,
        "layer_name": layer_name,
        "raw_triples_in_layer": len(layer_triples[stop_index]),
        "clusters_in_layer": n_current,
        "cumulative_clusters_after_layer": len(seen_clusters),
        "new_clusters_vs_previous_cumulative": n_new,
        "reinforced_clusters_seen_before": len(reinforced_clusters),
        "new_cluster_ratio": rate(n_new, n_current),
        "reinforcement_ratio": rate(len(reinforced_clusters), n_current),

        "new_clusters_matching_gold": new_matches_gold,
        "new_clusters_matching_native_final": new_matches_final,
        "new_clusters_matching_any_reference": new_matches_any,
        "new_uncertain_reference_unmatched": n_new - new_matches_any,
        "new_likely_noise_single_layer": new_likely_noise,
        "new_persistent_unmatched_uncertain": new_persistent_unmatched,

        "marginal_gold_supported_rate": rate(new_matches_gold, n_new),
        "marginal_native_final_supported_rate": rate(new_matches_final, n_new),
        "marginal_any_reference_supported_rate": rate(new_matches_any, n_new),
        "marginal_uncertain_reference_unmatched_rate": rate(n_new - new_matches_any, n_new),
        "marginal_likely_noise_rate": rate(new_likely_noise, n_new),
        "marginal_persistent_unmatched_uncertain_rate": rate(new_persistent_unmatched, n_new),

        "new_gold_references_first_covered": len(newly_covered_gold),
        "new_native_final_references_first_covered": len(newly_covered_final),

        "cumulative_gold_references_covered": len(covered_gold_refs),
        "gold_reference_count": len(gold_triples),
        "cumulative_gold_reference_coverage": rate(len(covered_gold_refs), len(gold_triples)),

        "cumulative_native_final_references_covered": len(covered_final_refs),
        "native_final_reference_count": len(native_final_triples),
        "cumulative_native_final_reference_coverage": rate(len(covered_final_refs), len(native_final_triples)),
    })

    for cid in sorted(new_clusters):
        row = cluster_alignment_df[cluster_alignment_df["cluster_id"] == cid].iloc[0].to_dict()
        row.update({
            "introduced_at_layer": stop_index,
            "introduced_at_layer_name": layer_name,
            "category": (
                "confirmed_by_gold_and_native_final" if row["matches_gold"] and row["matches_native_final"] else
                "confirmed_by_gold_only" if row["matches_gold"] else
                "adopted_by_native_final_only" if row["matches_native_final"] else
                "likely_noise_single_layer" if row["likely_noise_single_layer"] else
                "persistent_unmatched_uncertain" if row["persistent_unmatched_uncertain"] else
                "uncertain_reference_unmatched"
            )
        })
        new_cluster_detail_rows.append(row)


incremental_summary_df = pd.DataFrame(summary_rows)
new_cluster_details_df = pd.DataFrame(new_cluster_detail_rows)

display(incremental_summary_df)
incremental_summary_df.to_csv(OUTPUT_DIR / "incremental_layer_contribution_summary.csv", index=False)
new_cluster_details_df.to_csv(OUTPUT_DIR / "incremental_new_cluster_details.csv", index=False)

print("Saved:", OUTPUT_DIR / "incremental_layer_contribution_summary.csv")
print("Saved:", OUTPUT_DIR / "incremental_new_cluster_details.csv")


## Per-relation marginal additions

In [ ]:

per_relation_rows = []

for stop_index in sorted(layer_cluster_ids):
    layer_name = incremental_summary_df.loc[
        incremental_summary_df["stop_index"] == stop_index, "layer_name"
    ].iloc[0]
    new_ids = set(new_cluster_details_df[
        new_cluster_details_df["introduced_at_layer"] == stop_index
    ]["cluster_id"].astype(int).tolist())

    new_df = cluster_alignment_df[cluster_alignment_df["cluster_id"].isin(new_ids)].copy()
    for pred, g in new_df.groupby("predicate"):
        n = len(g)
        per_relation_rows.append({
            "stop_index": stop_index,
            "layer_name": layer_name,
            "relation": pred,
            "new_clusters": n,
            "new_matching_gold": int(g["matches_gold"].sum()),
            "new_matching_native_final": int(g["matches_native_final"].sum()),
            "new_matching_any_reference": int(g["matches_any_reference"].sum()),
            "new_likely_noise_single_layer": int(g["likely_noise_single_layer"].sum()),
            "new_persistent_unmatched_uncertain": int(g["persistent_unmatched_uncertain"].sum()),
            "marginal_any_reference_supported_rate": float(g["matches_any_reference"].sum()) / n if n else 0.0,
            "marginal_likely_noise_rate": float(g["likely_noise_single_layer"].sum()) / n if n else 0.0,
        })

per_relation_incremental_df = pd.DataFrame(per_relation_rows).sort_values(["stop_index", "relation"])
display(per_relation_incremental_df)
per_relation_incremental_df.to_csv(OUTPUT_DIR / "incremental_per_relation_contribution_summary.csv", index=False)
print("Saved:", OUTPUT_DIR / "incremental_per_relation_contribution_summary.csv")


## Plots

In [ ]:

def save_plot(fig, name):
    out = OUTPUT_DIR / name
    fig.savefig(out, dpi=180)
    plt.show()
    print("Saved:", out)


# 1. Cumulative alignment with native final export and gold.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(
    incremental_summary_df["stop_index"],
    incremental_summary_df["cumulative_gold_reference_coverage"],
    marker="o",
    label="Cumulative gold-reference coverage",
)
if len(native_final_triples):
    ax.plot(
        incremental_summary_df["stop_index"],
        incremental_summary_df["cumulative_native_final_reference_coverage"],
        marker="o",
        label="Cumulative native-final coverage",
    )
ax.set_title("Cumulative reference coverage by layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Coverage")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
save_plot(fig, "cumulative_reference_coverage_by_layer.png")


# 2. New useful vs uncertain/noise clusters per layer.
plot_cols = [
    "new_clusters_matching_any_reference",
    "new_persistent_unmatched_uncertain",
    "new_likely_noise_single_layer",
]
plot_df = incremental_summary_df[["stop_index"] + plot_cols].set_index("stop_index")
fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("What each layer adds: confirmed vs uncertain/noise clusters")
ax.set_xlabel("Layer")
ax.set_ylabel("New clusters")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
save_plot(fig, "marginal_new_information_categories_by_layer.png")


# 3. Marginal supported/noise rates.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(
    incremental_summary_df["stop_index"],
    incremental_summary_df["marginal_any_reference_supported_rate"],
    marker="o",
    label="New clusters supported by gold or final export",
)
ax.plot(
    incremental_summary_df["stop_index"],
    incremental_summary_df["marginal_uncertain_reference_unmatched_rate"],
    marker="o",
    label="New reference-unmatched clusters",
)
ax.plot(
    incremental_summary_df["stop_index"],
    incremental_summary_df["marginal_likely_noise_rate"],
    marker="o",
    label="New likely-noise singleton clusters",
)
ax.set_title("Marginal quality of information added by each layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Rate among newly introduced clusters")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
save_plot(fig, "marginal_quality_rates_by_layer.png")


# 4. New native final references first covered.
if len(native_final_triples):
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(
        incremental_summary_df["stop_index"],
        incremental_summary_df["new_native_final_references_first_covered"],
    )
    ax.set_title("Native final export facts first introduced by each layer")
    ax.set_xlabel("Layer")
    ax.set_ylabel("New native-final references first covered")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    save_plot(fig, "native_final_references_first_introduced_by_layer.png")


## Compact paper table

In [ ]:

paper_cols = [
    "stop_index",
    "layer_name",
    "raw_triples_in_layer",
    "clusters_in_layer",
    "new_clusters_vs_previous_cumulative",
    "reinforced_clusters_seen_before",
    "new_clusters_matching_any_reference",
    "new_uncertain_reference_unmatched",
    "new_likely_noise_single_layer",
    "new_gold_references_first_covered",
    "new_native_final_references_first_covered",
    "cumulative_gold_reference_coverage",
    "cumulative_native_final_reference_coverage",
    "marginal_any_reference_supported_rate",
    "marginal_likely_noise_rate",
]

paper_table = incremental_summary_df[paper_cols].copy()
display(paper_table)

paper_table.to_csv(OUTPUT_DIR / "paper_incremental_layer_contribution_table.csv", index=False)
print("Saved:", OUTPUT_DIR / "paper_incremental_layer_contribution_table.csv")



## Interpretation guide

Use this notebook to support a claim like:

> Rather than measuring whether each layer directly increases final F1, we measure what information each layer introduces into the cumulative evidence pool. Later layers may add fewer new gold facts, but they may reinforce, normalize, or make final-output facts recoverable. Reference-unmatched additions are treated as uncertain; if they appear in only one layer and are not aligned with gold or the native final export, they are counted as likely noise.

Recommended terms:

- **Confirmed new information**: a newly introduced cluster that aligns with gold or the native NeoOLAF final export.
- **Native-final aligned information**: information that appears in the final NeoOLAF export and is first recoverable from a given layer.
- **Uncertain information**: newly introduced cluster not matched to gold or final export.
- **Likely noise**: uncertain cluster that appears in only one layer and is never reinforced later.

Important: “likely noise” is not a manually verified false positive. It is a reference-based diagnostic category.
